# 論文1（Gunji et al., Robust swarm of soldier crabs）の図をつくる

モデルは `model/MAM_paper1.py`（11/24 に `MAM_basic_v1.py` から分けたもの。同じ seed なら v1 と同じ軌跡になる）。

**使い方**
1. このノートブックは `研究/` をカレントディレクトリにして開く。
2. 最初の2つのセルを実行してから、作りたい図のセルだけを実行する。図は `資料用の図/論文1/` に PNG で保存される。
3. 図に使う設定は `PRESET`（1つだけ使う図）と `USE`（比べる図）で選ぶ。

**設定の組（`PRESETS`）。** 論文1 は、先生の答えが出るまで1つに決まっていない（検証レポート 0.3 節）。

| 名前 | 中身 | 特徴 |
|---|---|---|
| `"text"` | 文面どおり（α そのまま・floor・基本ベクトルの向きで速度一致） | 整列度が 0.18 より下がらない |
| `"A"` | 候補A：α 2倍・round | 整列度は α≥0.4 で論文に合う。θ が完全にそろい、動きが固い |
| `"B"` | 候補B：α そのまま・round・移動ベクトルで速度一致 | α=0.3 の整列度と図13・14 の密度（Ls=10）が合う。群れがいくつかに分かれる |

文面と違うのは「α 2倍」と「round」（検証レポート 0.4 節）。**資料には、どの設定で描いたかを必ず書く。**

**かかる時間の目安**（この PC で。1回 600 ステップが P=15 で約 8 秒、P=30 で約 18 秒）：スナップショット 1 分、P を変える図 10〜15 分、α を変える図 5 分、スケールフリー相関 1 分。

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = ["Yu Gothic", "Meiryo", "MS Gothic", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

# True にすると、ステップ数とシード数を減らして素早く動作確認できる（図の値は信用しない）
QUICK = False

def steps(n):
    return max(60, n // 10) if QUICK else n

def seeds(n):
    return 1 if QUICK else n

def save(fig, name):
    """図を OUT に保存する（PNG、200 dpi）"""
    path = os.path.join(OUT, name)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("保存:", path)

def tic():
    return time.time()

def toc(t0, what=""):
    print(f"{what} {time.time() - t0:.0f} 秒")

from model.MAM_paper1 import (CrabSwarmPaper1, PRESETS, PRESET_NAMES, simulate,
                              polarization1, density_grid, correlation_function, torus_diff)

OUT = os.path.join("資料用の図", "論文1")
os.makedirs(OUT, exist_ok=True)

PRESET = "A"          # 1つの設定だけで描く図に使う
USE = ["A", "B"]      # 設定を比べる図に使う（"text" を足してもよい）
COLORS = {"text": "gray", "A": "tab:blue", "B": "tab:orange"}

# 論文の値（図を目で読んだもの。検証レポート 2.8・2.9 節）
FIG11_POL = {0.3: 0.53, 0.4: 0.30, 0.5: 0.13, 0.6: 0.10}                         # 図11 の整列度（P=10〜20）
FIG14_POL = [((0.2, 0.3), 0.53), ((0.3, 0.4), 0.35), ((0.4, 0.5), 0.20), ((0.5, 0.6), 0.08)]  # 図14（α の幅ごと）
FIG11_DENS = {0.3: {1.5: 0.31, 4: 0.89, 6: 0.80, 8: 0.62, 10: 0.67, 12: 0.65, 14: 0.61, 15: 0.70, 17: 0.78,
                    19: 0.86, 21: 0.88, 24: 0.91, 30: 0.93},
              0.6: {1.5: 0.32, 4: 0.83, 6: 0.82, 8: 0.74, 10: 0.75, 12: 0.73, 13: 0.77, 15: 0.82, 18: 0.85,
                    20: 0.88, 24: 0.91, 30: 0.95}}                                # 図11 の密度（区画 Ls=25）


def draw_swarm(ax, m, title=""):
    """個体の位置と、基本ベクトルの向き（矢印）を描く"""
    x, y = m.position[:, 0], m.position[:, 1]
    ax.quiver(x, y, np.cos(m.theta), np.sin(m.theta), angles="xy", scale_units="xy", scale=0.4,
              width=0.003, headwidth=4, color="k")
    ax.scatter(x, y, s=8, color="tab:red", zorder=3)
    ax.set_xlim(-1, m.S); ax.set_ylim(-1, m.S); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=10)

## 1. スナップショット（図8 のような P × α の並び）

`T` ステップ回したあとの位置と基本ベクトルの向き。P=1 は全員がそろって直進、P=2 で群れが崩れる（論文1 図8、検証レポート 2.1・2.2 節）。

In [ ]:
Ps = [1, 2, 10, 30]
alphas = [0.2, 0.5]
T = steps(300)

t0 = tic()
fig, axes = plt.subplots(len(alphas), len(Ps), figsize=(3 * len(Ps), 3 * len(alphas)))
for i, a in enumerate(alphas):
    for j, P in enumerate(Ps):
        m = CrabSwarmPaper1(P=P, alpha=a, seed=0, **PRESETS[PRESET])
        m.run(T)
        draw_swarm(axes[i, j], m, f"P={P}, α={a}")
fig.suptitle(f"論文1：t={T} のスナップショット（{PRESET_NAMES[PRESET]}）")
fig.tight_layout()
save(fig, f"01_スナップショット_{PRESET}.png")
toc(t0, "かかった時間")

## 2. 1本のシミュレーションの時間変化

整列度（変位の平均の長さ / L）と密度（区画 Ls の中の最大個体数 / N）の時系列。`USE` の設定を重ねる。

In [ ]:
P, a, T = 15, 0.3, steps(900)
Ls = 10

t0 = tic()
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for name in USE:
    pol, dens, rules = simulate(T, Ls=Ls, P=P, alpha=a, seed=0, **PRESETS[name])
    ax[0].plot(pol, color=COLORS[name], lw=0.8, label=PRESET_NAMES[name])
    ax[1].plot(dens, color=COLORS[name], lw=0.8, label=PRESET_NAMES[name])
ax[0].set_ylabel("整列度"); ax[1].set_ylabel(f"密度（区画 Ls={Ls}）")
for x in ax:
    x.set_xlabel("ステップ"); x.set_ylim(0, 1); x.legend(fontsize=8)
fig.suptitle(f"論文1：P={P}, α={a}（seed 0）")
fig.tight_layout()
save(fig, "02_時間変化.png")
toc(t0, "かかった時間")

## 3. P を変えたときの整列度・密度（図10右・図11）

横軸は P（論文の横軸「内部の揺らぎ」は P/30）。密度は図11 に合う「区画 Ls=25」（検証レポート 2.9 節）。各シードで `burn` ステップ以降を時間平均し、シードの平均と範囲を描く。黒い菱形は論文の図11 の密度。

In [ ]:
Ps = [1, 2, 3, 5, 8, 10, 15, 20, 30]
alphas = [0.3, 0.6]
T, burn, n_seed = steps(600), steps(200), seeds(2)

t0 = tic()
res = {}   # (設定, α, P) -> [(整列度, 密度), ...]
for name in USE:
    for a in alphas:
        for P in Ps:
            vals = []
            for s in range(n_seed):
                pol, dens, _ = simulate(T, Ls=25, P=P, alpha=a, seed=s, **PRESETS[name])
                vals.append((pol[burn:].mean(), dens[burn:].mean()))
            res[(name, a, P)] = np.array(vals)
toc(t0, "計算")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name in USE:
    for a, ls in zip(alphas, ["-", "--"]):
        v = np.array([res[(name, a, P)] for P in Ps])          # (P, seed, 2)
        for k in range(2):
            ax[k].plot(Ps, v[:, :, k].mean(1), ls, marker="o", ms=4, color=COLORS[name],
                       label=f"{PRESET_NAMES[name]}, α={a}")
            ax[k].fill_between(Ps, v[:, :, k].min(1), v[:, :, k].max(1), color=COLORS[name], alpha=0.12)
for a, mk in zip(alphas, ["D", "s"]):
    ax[1].plot(list(FIG11_DENS[a]), list(FIG11_DENS[a].values()), mk, color="k", ms=4, mfc="none",
               label=f"論文 図11, α={a}")
ax[0].set_ylabel("整列度"); ax[1].set_ylabel("密度（区画 Ls=25）")
for x in ax:
    x.set_xlabel("P（可能的遷移の数）"); x.set_ylim(0, 1); x.legend(fontsize=7)
fig.suptitle("論文1：P を変えたとき（各シード t≥burn の平均。帯はシードの範囲）")
fig.tight_layout()
save(fig, "03_Pを変える.png")

## 4. α を変えたときの整列度・密度（図11・図13・図14）

P=10〜20 のどれか（既定は 15）で α を振る。黒い菱形は図11 の整列度、灰色の横棒は図14（α の幅ごとの分布の中心）。密度は図13・14 の Ls が分からないので Ls=10 で描く（Ls≈6 なら候補A、Ls=10 なら候補B が合う。検証レポート 2.9 節）。

In [ ]:
P = 15
alphas = [0.2, 0.3, 0.4, 0.5, 0.6]
T, burn, n_seed = steps(600), steps(200), seeds(3)
Ls = 10

t0 = tic()
res_a = {}
for name in USE:
    for a in alphas:
        vals = []
        for s in range(n_seed):
            pol, dens, _ = simulate(T, Ls=Ls, P=P, alpha=a, seed=s, **PRESETS[name])
            vals.append((pol[burn:].mean(), dens[burn:].mean()))
        res_a[(name, a)] = np.array(vals)
toc(t0, "計算")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name in USE:
    v = np.array([res_a[(name, a)] for a in alphas])
    for k in range(2):
        ax[k].plot(alphas, v[:, :, k].mean(1), marker="o", color=COLORS[name], label=PRESET_NAMES[name])
        ax[k].fill_between(alphas, v[:, :, k].min(1), v[:, :, k].max(1), color=COLORS[name], alpha=0.15)
ax[0].plot(list(FIG11_POL), list(FIG11_POL.values()), "D", color="k", label="論文 図11")
for i, ((lo, hi), val) in enumerate(FIG14_POL):
    ax[0].hlines(val, lo, hi, color="gray", lw=4, alpha=0.6, label="論文 図14" if i == 0 else None)
ax[0].set_ylabel("整列度"); ax[1].set_ylabel(f"密度（区画 Ls={Ls}）")
for x in ax:
    x.set_xlabel("α"); x.set_ylim(0, 1); x.legend(fontsize=8)
fig.suptitle(f"論文1：α を変えたとき（P={P}、帯はシードの範囲）")
fig.tight_layout()
save(fig, "04_αを変える.png")

## 5. スケールフリー相関（図15）

揺らぎ u_i = v_i − (平均) の相関 C(r) を、相関長 ξ（C が 0 になる距離）で割った r/ξ に対して描く。図15 と同じ L=4、R(Nf)=2、**R(M)=2**。C は自分との積の平均で割る（Cavagna et al. 2010 と同じ）。

**速度 v_i を何で取るかで結果が大きく変わる**（検証レポート 2.13 節）。
- 移動（1ステップ）：傾きは論文の 1/10 くらい。
- 基本ベクトル L·(cosθ, sinθ)（論文 p.5 が「basic vector (velocity)」と呼ぶもの）：文面どおりの設定で傾き −1.06 と論文（−0.96）に合う。ただし θ はほとんどそろっていて、とても小さな揺らぎを測っている。候補A では θ が完全にそろって計算できないことが多い。

破線は論文の図15d（モデル）、点線は図15c（実際のカニ）。どちらも目で読んだもの。

In [ ]:
SF_PRESET = "text"
P, a = 15, 0.3
warm, T, every = steps(500), steps(600), 3

t0 = tic()
m = CrabSwarmPaper1(P=P, alpha=a, R_M=2, seed=0, **PRESETS[SF_PRESET])
m.run(warm)
xs = {"移動（1ステップ）": [], "基本ベクトル θ": []}
ys = {k: [] for k in xs}
for t in range(T):
    p0 = m.position.copy()
    m.step()
    if t % every:
        continue
    vels = {"移動（1ステップ）": torus_diff(m.position - p0, m.S).astype(float),
            "基本ベクトル θ": m.L * np.stack([np.cos(m.theta), np.sin(m.theta)], 1)}
    for k, v in vels.items():
        r = correlation_function(p0.astype(float), v, m.S)
        if r is None or np.isnan(r[2]):
            continue
        xs[k].append(r[0] / r[2]); ys[k].append(r[1])
toc(t0, "計算")

fig, ax = plt.subplots(figsize=(5.5, 4.2))
bins = np.arange(0, 1.65, 0.1)
for (k, col) in zip(xs, ["tab:blue", "tab:red"]):
    if not xs[k]:
        print(f"{k}：揺らぎが 0 で計算できなかった")
        continue
    x = np.concatenate(xs[k]); y = np.concatenate(ys[k])
    sel = x <= 1.2
    slope, icpt = np.polyfit(x[sel], y[sel], 1)
    idx = np.digitize(x, bins) - 1
    prof = [y[idx == i].mean() if np.sum(idx == i) >= 3 else np.nan for i in range(len(bins) - 1)]
    ax.plot((bins[:-1] + bins[1:]) / 2, prof, color=col, lw=2,
            label=f"{k}：傾き {slope:+.2f}（{len(xs[k])} 枚）")
xx = np.linspace(0, 1.4, 10)
ax.plot(xx, 1.05 - 0.96 * xx, "k--", label="論文 図15d（モデル）")
ax.plot(xx, 0.88 - 0.88 * xx, "k:", label="論文 図15c（カニ）")
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlim(0, 1.5); ax.set_ylim(-0.4, 1.2)
ax.set_xlabel("r / ξ"); ax.set_ylabel("C（自分との積で割る）")
ax.set_title(f"論文1：スケールフリー相関（{PRESET_NAMES[SF_PRESET]}、P={P}, α={a}, R(M)=2）", fontsize=9)
ax.legend(fontsize=8)
fig.tight_layout()
save(fig, f"05_スケールフリー相関_{SF_PRESET}.png")